# Notebook 01: Load and quality-check (Section 7)

Thin notebook: all logic lives in `pasig/`. Outputs: `data/interim/{waterlevel_long,pumping_long,pagasa_6h,pagasa_daily}.parquet`, `data/processed/{hourly_master,eval_hours}.parquet`, coverage tables, logs. Gates G1, G2, G3, G5, G6.

For the full run as a script use `python scripts/run_phase.py --phase 1 --tag`.

**Runs locally:** the raw data are committed in `data/raw/`; the first cell checks them against the manifest and, in a fresh clone, rebuilds any missing generated inputs (no Google Drive needed).

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from pasig.local import prepare
prepare(1)   # raw data from the repo (no Drive); rebuilds missing local inputs once per clone
import pandas as pd
from pasig import phase1
from pasig.gates import require_gates, read_gate
require_gates()  # Phase 1 needs no earlier gate

## 1.1 Manifest (G1)

In [ ]:
man, bl = phase1.step_manifest()
print(read_gate('G1')['evidence']['status_counts'])
man[man.status.ne('use')][['path','status','reason']]

## 1.2-1.3 EFCOS and pumping parsers (G2)

In [ ]:
selftest = phase1.rollover_selftest(); print(selftest)
wl = phase1.step_waterlevel(man)
pump = phase1.step_pumping(bl)
phase1.g2_gate(wl, pump, selftest)
wl['hl_by_file'].sort_values('rate', ascending=False).head(6)

## 1.4 PAGASA Science Garden (G3)

In [ ]:
pag = phase1.step_pagasa(); read_gate('G3')['decision'], pag['g3']

## 1.5-1.6 QC and hand check

In [ ]:
qc = phase1.step_qc(wl['long'], pump['long'], pag['six'])
qc['removals'].query('wet').groupby(['station','split','qc_flag']).size().unstack(fill_value=0)

In [ ]:
qc['handcheck'][['value_m','qc_flag','pandacan_t-1','pandacan_t+1','fort_santiago_t','aviles_outside_t','sg_rain_24h','verdict']]

## 1.7-1.8 Hourly master, coverage and eligibility (G5)

In [ ]:
cov = phase1.step_coverage(qc['H'], qc['H_sto_nino_no_exemption'])
master = phase1.hourly_master(qc['H'], pag['six'], pag['daily'])
master.to_parquet('data/processed/hourly_master.parquet')
display(cov['coverage'].round(3)); display(cov['coverage_no_exemption'].round(3)); cov['eligible']

![coverage](../reports/figures/fig_III-3_coverage_wet_season.png)

## 1.9 Evaluation hours and Table III-6

In [ ]:
ev = phase1.eval_hours(master, cov['eligible'])
ev.to_parquet('data/processed/eval_hours.parquet')
t6 = phase1.table_iii6(master, ev); t6.to_csv('reports/tables/table_III-6_recomputed.csv', index=False); t6.set_index('split').T

## 1.10 Tests (G6). Stop here if any fail; then `git tag phase1-frozen`.

In [ ]:
import subprocess
code = subprocess.call([sys.executable, '-m', 'pytest', '-q', 'tests/test_parsers.py', 'tests/test_time_alignment.py',
                       'tests/test_qc.py', 'tests/test_splits.py', 'tests/test_gapfill.py', 'tests/test_gates.py'])
assert phase1.g6_gate(code, Path('data/processed/hourly_master.parquet'), Path('data/processed/eval_hours.parquet')), 'G6 failed'